<a href="https://colab.research.google.com/github/manaspathak2335-git/Gridlock/blob/main/Manas%20Pathak/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pygeohash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00


In [2]:
#importing required libraries
from google.colab import drive
import pandas as pd
import pygeohash as pgh
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn import metrics

In [3]:
#mounting drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
#reading raw training dataset
raw_data = pd.read_csv('/content/drive/MyDrive/hackathon data/DATA/dataset/train.csv')
print(raw_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB
None


In [27]:
#reading raw testing dataset
test_data = pd.read_csv('/content/drive/MyDrive/hackathon data/DATA/dataset/test.csv')
print(test_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41454 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    40429 non-null  float64
 9   Weather        41347 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB
None


**PREPROCESSING**

In [28]:
#defining preprocessing function
def preprocess_data(df):

  #encoding day column
  df['day']=df['day'].map({48:0,49:1})

  #extracting longitude and latitude info from geohash column
  def decode_lat(hash_str):
    if pd.isna(hash_str):
      return None
    return pgh.decode(hash_str)[0]
  def decode_long(hash_str):
    if pd.isna(hash_str):
      return None
    return pgh.decode(hash_str)[1]
  df['latitude'] = df['geohash'].apply(decode_lat)
  df['longitude'] = df['geohash'].apply(decode_long)

  #encoding timestamp column
  def hour(ts):
    h,_ = map(int, ts.split(':'))
    return h
  df['hour'] = df['timestamp'].apply(hour)

  #encoding RoadType column
  df['RoadType'] = df['RoadType'].fillna('Unknown')
  df = pd.get_dummies(df, columns=['RoadType'], drop_first=False, dtype=int)

  #encoding LargeVehicles column
  df['LargeVehicles'] = df['LargeVehicles'].map({'Allowed':1, 'Not Allowed':0})

  #encoding Landmarks column
  df['Landmarks'] = df['Landmarks'].map({'Yes':1, 'No':0})

  #adressing missing values in temprature columns
  df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

  #encoding weather column
  df['Weather'] = df['Weather'].fillna('Unknown')
  df = pd.get_dummies(df, columns=['Weather'], drop_first=False, dtype=int)

  #dropping irrelevent columns
  df.drop(['Index','geohash','timestamp'], axis=1, inplace=True)

  return df

**Training baseline model**

In [29]:
#preprocessing training and testing datasets
train_data = preprocess_data(raw_data)
test_data_clean = preprocess_data(test_data)

In [30]:
print(train_data)
print("*"*50)
print(test_data_clean)

       day    demand  NumberofLanes  LargeVehicles  Landmarks  Temperature  \
0        0  0.048804              1              0          0    16.382587   
1        0  0.118507              3              1          1    31.104565   
2        0  0.027132              1              0          0    25.919267   
3        0  0.003272              1              0          0    16.382587   
4        0  0.010819              1              0          0    10.803667   
...    ...       ...            ...            ...        ...          ...   
77294    1  0.067203              1              0          0    11.501664   
77295    1  0.022859              3              1          1    14.715254   
77296    1  0.141342              3              1          1    19.678860   
77297    1  0.087574              1              0          0    22.573958   
77298    1  0.002944              3              1          1     1.322034   

       latitude  longitude  hour  RoadType_Highway  RoadType_Re

In [31]:
#splitting into X_train and y_train
X_train = train_data.drop(['demand'], axis=1)
y_train = train_data['demand']

In [32]:
#training xgboost model
xgb_regressor_v1 = XGBRegressor(random_state=42, objective='reg:squarederror', n_estimators = 800)
xgb_regressor_v1.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=800,
             n_jobs=None, num_parallel_tree=None, ...)

In [35]:
#making predictions
y_pred = xgb_regressor_v1.predict(test_data_clean)
y_pred_v1 = pd.DataFrame({'Index':test_data['Index'], 'demand':y_pred})
y_pred_v1.to_csv('/content/drive/MyDrive/hackathon data/DATA/dataset/prediction_v1.csv',index=False)